# GEDI L4B gridded aboveground biomass over the Ogallala aquifer at its own 1 km grid

NASA's GEDI Level 4B product (Version 2.1, ORNL DAAC): mean aboveground biomass density on a 1 km grid, made
by the GEDI science team from the laser shots of April 2019 to March 2023. It is one map for that whole
period, not a time series. The notebook downloads NASA's ten GeoTIFF layers, cuts each to the aquifer on
the product's own grid, and writes them into one CF NetCDF file on Drive.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/GEDI_L4B_v2_1_1km/GEDI_L4B_v2_1_1km_Ogallala.nc` |
| Grid | the product's own 1 km grid and projection, read from the files. **Nothing is resampled or reprojected** |
| Cells kept | cells whose centre lies inside the aquifer polygon; the others hold the layer's own "no data" value |
| Values | exactly the numbers in NASA's files, in their own data type |

**The ten layers** (the two-letter names are NASA's; the meanings are given as I know them from the product's
user guide and should be checked against it before publication):

| Layer | Meaning |
|---|---|
| `MU` | mean aboveground biomass density (Mg/ha) |
| `SE` | standard error of the mean (Mg/ha) |
| `PE` | standard error as a percentage of the mean |
| `V1`, `V2` | the two variance components behind the standard error |
| `QF` | quality flag |
| `NC` | number of tracks (clusters) with shots in the cell |
| `NS` | number of shots in the cell |
| `MI` | mode of inference |
| `PS` | prediction stratum |

Whatever description, units and tags the GeoTIFFs themselves carry are copied into the file as attributes.

**How to run**

1. Colab → key icon (Secrets) → add `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` (your NASA Earthdata login)
   and allow this notebook to read them. The notebook reads them from there and nowhere else.
2. *Run all*. Ten files, 2.5 GB in all, are downloaded once to Colab's scratch disk and
   read there; the file written to Drive is small.
3. Running it again does nothing if the file on Drive was made from the same NASA files; set
   `FORCE = True` to make it again.

**What the notebook checks**

* Every downloaded file must have the size and the SHA-256 checksum NASA's catalogue gives.
* All layers must be on one grid; otherwise the run stops.
* After writing, every variable is read back from the NetCDF file and compared with a fresh reading of
  the GeoTIFF. Only then is the file copied to Drive, with a checksum comparison.

**What to know before using the data**

* This is a 1 km product. A cell's value is the GEDI team's estimate of the mean biomass of the cell, from
  the shots that fell in it and, where shots are few, from a model; `MI`, `NS` and `NC` say which.
* GEDI measures woody vegetation structure. Over cropland and grassland, which is most of the High Plains,
  biomass is low and the relative error (`PE`) is large; use `SE`, `PE` and `QF` when interpreting `MU`.
* The map describes 2019-2023 as a whole. It cannot show change within that period.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# numpy, geopandas, shapely and pyproj are part of Colab.
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q earthaccess rasterio netCDF4 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
SHORT_NAME = "GEDI_L4B_Gridded_Biomass_V2_1_2299"   # NASA's name of the collection (ORNL DAAC)
VERSION    = "2.1"
LAYERS     = None          # None -> every layer NASA provides; or e.g. ["MU", "SE", "QF"]
PAD_CELLS  = 1             # cells kept around the aquifer's bounding box
FORCE      = False         # True -> make the file again even if Drive has it from the same NASA files
KEEP_DOWNLOADS = False     # True -> leave NASA's GeoTIFFs on Colab's scratch disk after the run

COMPRESS_LEVEL = 2         # zlib level of the .nc file (lossless)
MIN_FREE_GB    = 10.0      # scratch space that must stay free
DOWNLOAD_TRIES = 4         # attempts per file
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, Earthdata sign-in, Drive copy ═══════════════
import re, sys, json, time, shutil, hashlib, logging, warnings
from datetime import datetime, timezone

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import rasterio
import rasterio.features
import rasterio.windows
from affine import Affine
import earthaccess

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("gedi_l4b")
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")

ROI_LABEL   = "Ogallala"
PRODUCT_TAG = "GEDI_L4B_v" + VERSION.replace(".", "_") + "_1km"
DRIVE_OUT   = f"/content/drive/MyDrive/MSUGWB/{PRODUCT_TAG}"
OUT_NAME    = f"{PRODUCT_TAG}_{ROI_LABEL}.nc"
SCRATCH     = "/content/gedi_l4b_scratch"
RAW_DIR     = f"{SCRATCH}/raw"
for _d in (DRIVE_OUT, RAW_DIR):
    os.makedirs(_d, exist_ok=True)

# NASA's two-letter layer names. The meanings are as I know them from the product's user guide.
LAYER_INFO = dict(
    MU=("mean aboveground biomass density", "Mg ha-1"),
    SE=("standard error of the mean aboveground biomass density", "Mg ha-1"),
    PE=("standard error as a percentage of the mean", "percent"),
    V1=("variance component 1 of the standard error", ""),
    V2=("variance component 2 of the standard error", ""),
    QF=("quality flag", "1"),
    NC=("number of tracks (clusters) with shots in the cell", "1"),
    NS=("number of shots in the cell", "1"),
    MI=("mode of inference", "1"),
    PS=("prediction stratum", "1"))


class PipelineStop(RuntimeError):
    """Clean stop with a message; nothing on Drive was changed."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


def free_gb():
    return shutil.disk_usage("/content").free / 1024 ** 3


def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def drive_copy(src, dst, tries=3):
    """Copy to Drive under a temporary name, read it back, compare size and MD5, then rename."""
    size, want, tmp = os.path.getsize(src), md5sum(src), dst + ".tmp"
    last = None
    for attempt in range(1, tries + 1):
        try:
            shutil.copyfile(src, tmp)
            time.sleep(DRIVE_SETTLE_S)
            if os.path.getsize(tmp) != size or md5sum(tmp) != want:
                raise IOError("size or MD5 mismatch after the copy")
            os.replace(tmp, dst)
            log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
            return
        except Exception as e:
            last = e
            log.warning("Drive copy attempt %d/%d failed: %s", attempt, tries, e)
            if os.path.exists(tmp):
                os.remove(tmp)
            time.sleep(10 * attempt)
    raise PipelineStop(f"The file could not be saved to Drive ({last}). Run again.")


DRIVE_SETTLE_S = 10


def earthdata_login():
    """Sign in to NASA Earthdata with the login kept in Colab Secrets. The notebook never stores it."""
    try:
        from google.colab import userdata
        for key in ("EARTHDATA_USERNAME", "EARTHDATA_PASSWORD"):
            try:
                val = userdata.get(key)
                if val:
                    os.environ[key] = val
            except Exception:
                pass
    except ImportError:
        pass
    if not (os.environ.get("EARTHDATA_TOKEN") or (os.environ.get("EARTHDATA_USERNAME")
                                                  and os.environ.get("EARTHDATA_PASSWORD"))):
        log.info("No Colab Secret found - earthaccess will try ~/.netrc, then ask.")
    auth = earthaccess.login()
    if not getattr(auth, "authenticated", False):
        raise PipelineStop("Earthdata login failed - check EARTHDATA_USERNAME / EARTHDATA_PASSWORD in Colab Secrets.")
    log.info("Signed in to NASA Earthdata")

In [ ]:
# ═══════════════ CELL 4: find NASA's files, download, cut to the aquifer, write, check, copy to Drive ═══════════════
def find_layers():
    """NASA's GeoTIFFs of the collection, from its catalogue: {layer: dict(name, url, size)}."""
    out = {}
    for g in earthaccess.search_data(short_name=SHORT_NAME, version=VERSION, count=200):
        for url in g.data_links(access="external"):
            name = url.split("/")[-1]
            m = re.fullmatch(r"(.+)_([A-Z][A-Z0-9])\.tif", name)
            if not m:
                continue
            files = g["umm"].get("DataGranule", {}).get("ArchiveAndDistributionInformation", [])
            one = files[0] if len(files) == 1 else next((d for d in files if d.get("Name") == name), {})
            check = one.get("Checksum", {})
            out[m.group(2)] = dict(name=name, url=url, size=one.get("SizeInBytes"),       # as NASA's catalogue states them
                                   sha256=check.get("Value") if str(check.get("Algorithm", "")).upper() in ("SHA-256", "SHA256") else None)
    if not out:
        raise PipelineStop(f"NASA's catalogue lists no GeoTIFF for {SHORT_NAME} version {VERSION}.")
    if LAYERS:
        absent = [k for k in LAYERS if k not in out]
        if absent:
            raise PipelineStop(f"Layers {absent} are not in the collection; it has {sorted(out)}.")
        out = {k: out[k] for k in LAYERS}
    return dict(sorted(out.items()))


def sha256sum(path, block=10 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def whole(path, info):
    """True when a downloaded GeoTIFF has the size and SHA-256 checksum NASA's catalogue states (where
    it states them) and opens and reads at both ends."""
    try:
        if info["size"] and os.path.getsize(path) != info["size"]:
            return False
        if info["sha256"] and sha256sum(path) != info["sha256"].lower():
            return False
        with rasterio.open(path) as ds:
            ds.read(1, window=rasterio.windows.Window(max(0, ds.width - 64), max(0, ds.height - 64), 64, 64))
            ds.read(1, window=rasterio.windows.Window(0, 0, 64, 64))
        return True
    except Exception:
        return False


def download(layers):
    """Every layer's file on scratch, complete. A file that is already there and whole is not fetched again."""
    for key, info in layers.items():
        info["path"] = os.path.join(RAW_DIR, info["name"])
        for attempt in range(1, DOWNLOAD_TRIES + 1):
            if os.path.exists(info["path"]) and whole(info["path"], info):
                break
            if os.path.exists(info["path"]):
                os.remove(info["path"])
            if free_gb() < MIN_FREE_GB:
                raise PipelineStop(f"Only {free_gb():.1f} GB free on the scratch disk; {MIN_FREE_GB} GB are needed.")
            log.info("Downloading %s (attempt %d)", info["name"], attempt)
            try:
                earthaccess.download([info["url"]], local_path=RAW_DIR)
            except Exception as e:
                log.warning("Download of %s failed: %s", info["name"], str(e)[:200])
                time.sleep(15 * attempt)
        else:
            raise PipelineStop(f"{info['name']} could not be downloaded completely in {DOWNLOAD_TRIES} attempts. Run again.")
        log.info("%s on scratch: %s, %s", info["name"], human(os.path.getsize(info["path"])),
                 "size and SHA-256 as in NASA's catalogue" if info["sha256"] else "opens and reads (the catalogue gives no checksum)")


def aquifer_window(ds, polygon):
    """The block of the file's own cells that bounds the polygon (plus PAD_CELLS), and its transform."""
    if ds.transform.b or ds.transform.d:
        raise PipelineStop("The GeoTIFF's grid is rotated; this notebook does not resample, so it stops.")
    left, bottom, right, top = polygon.bounds
    win = rasterio.windows.from_bounds(left, bottom, right, top, ds.transform)
    c0 = max(0, int(np.floor(win.col_off)) - PAD_CELLS)
    r0 = max(0, int(np.floor(win.row_off)) - PAD_CELLS)
    c1 = min(ds.width, int(np.ceil(win.col_off + win.width)) + PAD_CELLS)
    r1 = min(ds.height, int(np.ceil(win.row_off + win.height)) + PAD_CELLS)
    if c0 >= c1 or r0 >= r1:
        raise PipelineStop("The aquifer does not lie on the product's grid.")
    w = rasterio.windows.Window(c0, r0, c1 - c0, r1 - r0)
    return w, ds.window_transform(w)


def read_layers(layers):
    """Every layer's block over the aquifer, as stored. Returns (grid, mask, {layer: dict(data, ...)})."""
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs is None:
        raise PipelineStop(f"{SHP_PATH} has no CRS")
    grid, mask, out = None, None, {}
    for key, info in layers.items():
        with rasterio.open(info["path"]) as ds:
            if ds.count != 1:
                raise PipelineStop(f"{info['name']} has {ds.count} bands; one was expected.")
            this = (ds.crs.to_wkt(), tuple(ds.transform)[:6], ds.width, ds.height)
            if grid is None:
                geom = gdf.geometry
                merged = geom.union_all() if hasattr(geom, "union_all") else geom.unary_union
                polygon = gpd.GeoSeries([merged], crs=gdf.crs).to_crs(ds.crs).iloc[0]
                polygon = polygon if polygon.is_valid else polygon.buffer(0)
                win, transform = aquifer_window(ds, polygon)
                grid = dict(id=this, crs=ds.crs, window=win, transform=transform, h=int(win.height), w=int(win.width),
                            full=(ds.width, ds.height))
                mask = rasterio.features.geometry_mask([polygon], out_shape=(grid["h"], grid["w"]), transform=transform,
                                                       invert=True, all_touched=False)       # cell centre inside the polygon
            elif this != grid["id"]:
                raise PipelineStop(f"{info['name']} is on another grid than the first layer. Nothing was written.")
            data = ds.read(1, window=grid["window"])
            tags = {k: str(v) for k, v in {**ds.tags(), **ds.tags(1)}.items()}
            out[key] = dict(data=data, nodata=ds.nodata, dtype=data.dtype, tags=tags, description=ds.descriptions[0],
                            units=ds.units[0], scale=ds.scales[0], offset=ds.offsets[0], name=info["name"],
                            bytes=os.path.getsize(info["path"]))
    log.info("Grid: %s, cell %.3f x %.3f, block over the aquifer %d x %d cells, %d of them inside the polygon",
             grid["crs"].to_string(), grid["transform"].a, -grid["transform"].e, grid["h"], grid["w"], int(mask.sum()))
    return grid, mask, out


def blanked(rec, mask):
    """The layer with cells outside the polygon set to its own "no data" value (kept as they are when
    the file names none: roi_mask then says which cells are inside)."""
    if rec["nodata"] is None:
        return rec["data"]
    return np.where(mask, rec["data"], np.asarray(rec["nodata"]).astype(rec["dtype"]))


def write_nc(path, grid, mask, recs):
    crs = pyproj.CRS.from_wkt(grid["crs"].to_wkt())
    t = grid["transform"]
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    tmp = path + ".new"
    with nc4.Dataset(tmp, "w", format="NETCDF4") as ds:
        ds.createDimension("y", grid["h"])
        ds.createDimension("x", grid["w"])
        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis, yv.long_name = "projection_y_coordinate", "m", "Y", "y of cell centre"
        yv[:] = t.f + (np.arange(grid["h"]) + 0.5) * t.e
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis, xv.long_name = "projection_x_coordinate", "m", "X", "x of cell centre"
        xv[:] = t.c + (np.arange(grid["w"]) + 0.5) * t.a
        if crs.is_geographic:
            yv.standard_name, yv.units, xv.standard_name, xv.units = "latitude", "degrees_north", "longitude", "degrees_east"
        gm = ds.createVariable("crs", "i4")
        try:
            for k, v in crs.to_cf().items():
                gm.setncattr(k, v if isinstance(v, (str, int, float)) else np.asarray(v) if isinstance(v, (list, tuple)) else str(v))
        except Exception as e:
            log.warning("CF grid-mapping attributes could not be derived (%s); the WKT is stored", e)
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{t.c} {t.a} {t.b} {t.f} {t.d} {t.e}"
        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), **zkw)
        mv.long_name = f"1 where the cell centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values, mv.flag_meanings, mv.grid_mapping = np.array([0, 1], dtype="i1"), "outside_roi inside_roi", "crs"
        mv[:] = mask.astype("i1")
        for key, rec in recs.items():
            kw = {} if rec["nodata"] is None else dict(fill_value=np.asarray(rec["nodata"]).astype(rec["dtype"])[()])
            v = ds.createVariable(key, rec["dtype"], ("y", "x"), **kw, **zkw)
            v.set_auto_maskandscale(False)
            meaning, units = LAYER_INFO.get(key, (f"layer {key} of the product", ""))
            v.long_name = meaning
            if rec["units"] or units:
                v.units = rec["units"] or units
            v.grid_mapping = "crs"
            v.source_file = rec["name"]
            v.source_file_bytes = np.int64(rec["bytes"])
            if rec["description"]:
                v.source_description = rec["description"]
            if rec["scale"] not in (None, 1.0) or rec["offset"] not in (None, 0.0):
                v.scale_factor, v.add_offset = rec["scale"], rec["offset"]
            if rec["tags"]:
                v.source_tags = json.dumps(rec["tags"], sort_keys=True)
            v.comment = ("Exactly the values of NASA's GeoTIFF on its own grid. Cells outside the polygon hold the "
                         "layer's own no-data value." if rec["nodata"] is not None else
                         "Exactly the values of NASA's GeoTIFF on its own grid. The GeoTIFF names no no-data value, so "
                         "cells outside the polygon keep their values; use roi_mask.")
            v[:] = blanked(rec, mask)
        ds.Conventions = "CF-1.9"
        ds.title = f"GEDI L4B gridded aboveground biomass density, version {VERSION}, {ROI_LABEL} aquifer, 1 km"
        ds.summary = ("NASA's GEDI Level 4B layers cut to the aquifer on the product's own grid. Nothing is resampled, "
                      "reprojected, scaled or filled. One estimate for the whole mission period of the product, not a "
                      "time series.")
        ds.institution = "NASA GEDI science team, ORNL DAAC (source data); packaged by the MSU groundwater project"
        ds.source = f"{SHORT_NAME} version {VERSION}, ORNL DAAC, retrieved with earthaccess"
        ds.references = "Dubayah et al., GEDI L4B Gridded Aboveground Biomass Density, Version 2.1, https://doi.org/10.3334/ORNLDAAC/2299"
        ds.history = f"{iso(utcnow())} created by GEDI_L4B_Biomass_1km_Ogallala.ipynb"
        ds.layers = " ".join(recs)
        ds.source_signature = signature(recs)
        ds.roi = ROI_LABEL
    os.replace(tmp, path)


def signature(recs):
    """What the file was made from: NASA's file names and sizes, and the settings that shape it."""
    return json.dumps(dict(files={k: [r["name"], int(r["bytes"])] for k, r in recs.items()}, pad=PAD_CELLS,
                           version=VERSION), sort_keys=True)


def check_nc(path, grid, mask, layers, recs):
    """Read every variable back from the closed file and compare it with a fresh reading of the GeoTIFF."""
    with nc4.Dataset(path) as ds:
        ds.set_auto_maskandscale(False)
        if not np.array_equal(np.asarray(ds["roi_mask"][:]).astype(bool), mask):
            raise PipelineStop("roi_mask in the file differs from the mask that was made. Nothing was copied to Drive.")
        for key, rec in recs.items():
            with rasterio.open(layers[key]["path"]) as src:
                fresh = dict(rec, data=src.read(1, window=grid["window"]))
            got, want = np.asarray(ds[key][:]), blanked(fresh, mask)
            if got.dtype != want.dtype or not np.array_equal(got, want, equal_nan=got.dtype.kind == "f"):
                raise PipelineStop(f"{key} in the file differs from NASA's GeoTIFF. Nothing was copied to Drive.")
    log.info("All %d layers read back from the file are identical to NASA's GeoTIFFs", len(recs))


def already_done(layers):
    """True when Drive holds a file made from exactly these NASA files (by name and catalogue size)."""
    path = os.path.join(DRIVE_OUT, OUT_NAME)
    if FORCE or not os.path.exists(path):
        return False
    try:
        with nc4.Dataset(path) as ds:
            sig = json.loads(ds.source_signature)
    except Exception as e:
        raise PipelineStop(f"{OUT_NAME} is on Drive but cannot be read ({e}). Nothing was changed. Run again in a few "
                           "minutes; if it really is damaged, move it away yourself and run again.")
    want = {k: v["name"] for k, v in layers.items()}
    have = {k: v[0] for k, v in sig.get("files", {}).items()}
    sizes_ok = all(layers[k]["size"] in (None, sig["files"][k][1]) for k in want if k in sig.get("files", {}))
    return have == want and sizes_ok and sig.get("pad") == PAD_CELLS


def run():
    t0 = time.time()
    try:
        earthdata_login()
        layers = find_layers()
        log.info("%d layers in NASA's catalogue: %s", len(layers), " ".join(layers))
        if already_done(layers):
            log.info("%s on Drive was made from these same NASA files - nothing to do (FORCE = True makes it again)", OUT_NAME)
            return os.path.join(DRIVE_OUT, OUT_NAME)
        download(layers)
        grid, mask, recs = read_layers(layers)
        local = os.path.join(SCRATCH, OUT_NAME)
        write_nc(local, grid, mask, recs)
        check_nc(local, grid, mask, layers, recs)
        drive_copy(local, os.path.join(DRIVE_OUT, OUT_NAME))
        os.remove(local)
        if not KEEP_DOWNLOADS:
            shutil.rmtree(RAW_DIR, ignore_errors=True)
        log.info("DONE in %.1f min: %s", (time.time() - t0) / 60, os.path.join(DRIVE_OUT, OUT_NAME))
        return os.path.join(DRIVE_OUT, OUT_NAME)
    except PipelineStop as e:
        print("\n" + "=" * 78 + f"\n  STOPPED\n  {e}\n" + "=" * 78 + "\n")
        return None


RESULT = run()

In [ ]:
# ═══════════════ CELL 5: what is on Drive ═══════════════
def describe(path):
    if not path or not os.path.exists(path):
        print("No output file.")
        return
    with nc4.Dataset(path) as ds:
        ds.set_auto_maskandscale(False)
        inside = np.asarray(ds["roi_mask"][:]).astype(bool)
        print(f"{path}\n{human(os.path.getsize(path))} | grid {len(ds.dimensions['y'])} x {len(ds.dimensions['x'])} cells | "
              f"{int(inside.sum()):,} cells inside the aquifer\n")
        print(f"{'layer':6s} {'type':8s} {'no data':>10s} {'cells with a value':>19s} {'min':>12s} {'mean':>12s} {'max':>12s}  meaning")
        for key in ds.layers.split():
            v = ds[key]
            a = np.asarray(v[:])[inside]
            nod = getattr(v, "_FillValue", None)
            ok = np.isfinite(a.astype("f8")) & ((a != nod) if nod is not None else True)
            a = a[ok].astype("f8")
            stats = (f"{a.min():12.3f} {a.mean():12.3f} {a.max():12.3f}" if a.size else f"{'-':>12s} {'-':>12s} {'-':>12s}")
            print(f"{key:6s} {str(v.dtype):8s} {str(nod):>10s} {a.size:19,d} {stats}  {v.long_name}")


describe(RESULT or os.path.join(DRIVE_OUT, OUT_NAME))

# Reading the file:
#   import xarray as xr
#   ds = xr.open_dataset("/content/drive/MyDrive/MSUGWB/GEDI_L4B_v2_1_1km/GEDI_L4B_v2_1_1km_Ogallala.nc")
#   agbd = ds.MU.where(ds.roi_mask == 1)           # Mg/ha; xarray turns the no-data value into NaN
#   good = ds.MU.where(ds.QF > 0)                  # check the meaning of QF in NASA's user guide first
# The grid is the product's own (see ds.crs.spatial_ref). To compare with the 30 m Landsat or OpenET
# tiles, reproject those to this grid in analysis; this file itself is never resampled.